# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [ ]:
ROLL_NO = 1024170424         # <-- your full roll number
NAME = "Kshitiz Poudel"            # <-- your name

import nltk

# Download the resources used in this notebook (needed in a fresh Colab runtime).
required_packages = [
    "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"
]
for pkg in required_packages:
    nltk.download(pkg, quiet=True)

assert ROLL_NO > 0 and NAME.strip(), "Enter your roll number and name first"

# Check the resources now so a download problem is reported here, not halfway through the lab.
resource_checks = {
    "stopwords": "corpora/stopwords",
    "Punkt sentence tokenizer": "tokenizers/punkt",
    "WordNet": "corpora/wordnet",
}
missing = []
for label, resource in resource_checks.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        # Some NLTK versions keep punkt_tab separately.
        if label == "Punkt sentence tokenizer":
            try:
                nltk.data.find("tokenizers/punkt_tab")
            except LookupError:
                missing.append(label)
        else:
            missing.append(label)
if missing:
    raise RuntimeError(
        "NLTK resources could not be downloaded: " + ", ".join(missing) +
        ". In Google Colab, enable internet access and run this cell again."
    )


### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

tickets = df['Ticket_Text'].tolist()
res = [process_ticket(t) for t in tickets]

rows = []
for tid, r in zip(df['Ticket_ID'], res):
    rows.append({'Ticket_ID': tid, 'sentences': len(r['sentences']), 'tokens': len(r['tokens']),
                 'punct': len(r['punct']), 'stopwords': len(r['stops']),
                 'unique_tokens': len(set(r['tokens'])), 'final_tokens': len(r['final'])})
tab = pd.DataFrame(rows)
tab.loc[len(tab)] = ['TOTAL'] + tab.drop(columns='Ticket_ID').sum().tolist()
display(tab)

k = 5
print(tickets[k])
print('tokens:', res[k]['tokens'])
print('punct:', res[k]['punct'])
print('stopwords:', res[k]['stops'])
print('final:', res[k]['final'])

**Answer:** In the final-token list, items such as `n't`, `wo`, or `'s` may appear even though they are not normal words by themselves. `word_tokenize()` separates some contractions into parts, and the pipeline only removes punctuation and words in the stopword list. It does not remove numbers or identifiers, so those can remain too. The examples for this assignment are the ones shown in the printed `final` list for the generated tickets.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

long_words = r'\b[A-Za-z]{6,}\b'
numbers = r'\b\d+(?:[.,]\d+)*\b'
capitalized = r'\b[A-Z][A-Za-z]*(?:-[A-Za-z]+)*\b'
vowel_words = r"\b[AEIOUaeiou][A-Za-z]*(?:'[a-z]+)?\b"
email = r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+'
url = r'(?:https?://|www\.)[^\s<>"]*[^\s<>".,;:!?)]'
phone = r'\+\d{1,3}[\s-]?\d{5}[\s-]?\d{5}|\b\d{3,5}-\d{3,4}-\d{4}\b'

def extract(text):
    return {'long': re.findall(long_words, text), 'numbers': re.findall(numbers, text),
            'capitalized': re.findall(capitalized, text), 'vowel': re.findall(vowel_words, text)}

def mask(text):
    text = re.sub(email, '<EMAIL>', text)
    text = re.sub(url, '<URL>', text)
    text = re.sub(phone, '<PHONE>', text)
    return text

for t in TEST_LINES + tickets:
    print(t)
    print(extract(t))
    print(mask(t))
    print()

total = sum(len(re.findall(capitalized, t)) for t in tickets)
start = sum(1 for t in tickets for s in nltk.sent_tokenize(t) if re.match(capitalized, s))
print(total, start)

**Answer:** A capital letter at the beginning of a sentence is not enough to show that a word is a name. The `total` count includes capitalized matches anywhere in the tickets, while `start` counts matches at the start of each sentence. The difference gives an estimate of capitalized words that occur elsewhere, although acronyms and product names can also affect it.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
pattern = '|'.join([url, email, phone, r'\d+(?:[.,]\d+)+', r'(?:[A-Za-z]\.){2,}',
                    r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*", r'[^\w\s]'])
tok = RegexpTokenizer(pattern)

def my_tokenize(text):
    return tok.tokenize(text)

for t in TEST_LINES + tickets:
    a, b = my_tokenize(t), nltk.word_tokenize(t)
    print(t)
    print('mine:', a)
    print('nltk:', b)
    print('same' if a == b else 'DIFFERENT')
    print()

**Answer:** The tickets marked `DIFFERENT` are the ones where the custom tokenizer and `word_tokenize()` split the text differently. This is expected because my pattern keeps items such as contractions, hyphenated terms, URLs, email addresses, and phone numbers together. The exact ticket count is shown by the comparison output for this roll number.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
porter = PorterStemmer()
lancaster = LancasterStemmer()
regstem = RegexpStemmer('ing$|ed$|s$', min=4)
lemm = WordNetLemmatizer()

def wn_pos(tag):
    if tag.startswith('J'): return 'a'
    if tag.startswith('V'): return 'v'
    if tag.startswith('R'): return 'r'
    return 'n'

tags = {}
for r in res:
    for w, tg in nltk.pos_tag(r['tokens']):
        tags.setdefault(w, tg)

uniq = sorted(set(w for r in res for w in r['final']))
out = pd.DataFrame({
    'word': uniq,
    'porter': [porter.stem(w) for w in uniq],
    'lancaster': [lancaster.stem(w) for w in uniq],
    'regexp': [regstem.stem(w) for w in uniq],
    'lemma': [lemm.lemmatize(w, wn_pos(tags[w])) for w in uniq]
})
diff = out[out[['porter', 'lancaster', 'regexp', 'lemma']].nunique(axis=1) > 1]
display(diff)

**Answer:** Over-stemming is when a word is shortened so much that its meaning or form is lost; under-stemming is when related words are not reduced to the same stem. The regular-expression stemmer here removes `ing`, `ed`, or `s` from words of at least four characters. I would use the comparison table above to point out one case where that rule makes an awkward form and one case where related forms still differ.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
S1 = [t for r in res for t in r['tokens']]
S2 = [t for t in S1 if not is_punct(t)]
S3 = [w for r in res for w in r['final']]
S4 = [porter.stem(w) for w in S3]

def stats(s):
    return len(s), len(set(s)), round(len(set(s)) / len(s), 4)

st = pd.DataFrame([stats(S1), stats(S2), stats(S3), stats(S4)],
                  index=['S1', 'S2', 'S3', 'S4'], columns=['total', 'unique', 'TTR'])
display(st)

print(Counter(S1).most_common(10))
print(Counter(S3).most_common(10))

freq = sorted(Counter(S3).values(), reverse=True)
ranks = np.arange(1, len(freq) + 1)
plt.plot(np.log(ranks), np.log(freq), marker='o')
plt.xlabel('log(rank)')
plt.ylabel('log(frequency)')
plt.title('Zipf plot for S3')
plt.show()

print(Counter(b for r in res for b in nltk.bigrams(r['tokens'])).most_common(5))
print(Counter(b for r in res for b in nltk.bigrams(r['final'])).most_common(5))

**Answer:** TTR is the number of unique tokens divided by the total number of tokens. From S1 to S3, punctuation and stopwords are removed, so the total gets smaller and the ratio will often rise. From S3 to S4, the token count stays the same, but Porter stemming can combine different word forms under one stem, which lowers the number of unique tokens. The table and frequency lists show how this works for these tickets.

## Q6.

**(a)** TTR is calculated as unique tokens divided by total tokens. Removing punctuation and stopwords reduces the corpus size; stemming can also reduce the number of unique forms by grouping related words together. The S1–S4 table above gives the values for this dataset.

**(b)** The custom tokenizer keeps contractions such as `isn't` together. The standard tokenizer can split a contraction into pieces, so the resulting token lists are not always the same. The same idea applies to URLs, email addresses, hyphenated words, and phone numbers.

**(c)** Stemming follows suffix rules instead of checking whether the result is a real dictionary word. That is why a Porter stem can look shortened or incomplete. The Porter column above can be used to identify an example from these tickets.

**(d)** Lemmatization uses a word's part of speech and a vocabulary to find its base form. It can therefore return a meaningful word where a stemmer returns a shortened form. The comparison table above shows examples from this dataset.